# Week 6 — ST-GCN: Spatio-Temporal Graph Convolutional Network

GNN looks at one month at a time and has no memory of what happened before. Adding a temporal dimension using LSTM layers makes it a Spatio-Temporal GCN (ST-GCN).


Same leakage-free features as the GNN: log_imports, lag exports, exchange rate, GDP, inflation. Train on 2017–2022, test on 2023–2024.

Anomalies flagged using per-country z-score > 2.0 on residuals.

In [1]:
%pip install torch-geometric


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from torch_geometric.nn import GCNConv

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "final").is_dir():
        return cwd
    if (cwd.parent / "data" / "final").is_dir():
        return cwd.parent
    return cwd


BASE = _repo_root()
device = torch.device("cpu")
SEQ_LEN = 6

print(f"PyTorch   : {torch.__version__}")
print(f"BASE      : {BASE}")
print(f"SEQ_LEN   : {SEQ_LEN} months")
print(f"Data file : {(BASE / 'data/final/master_trade_flow.csv').exists()}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch   : 2.11.0
BASE      : /Users/angollapraveengoud/Desktop/ONE_LAST!
SEQ_LEN   : 6 months
Data file : True


### Load Data and Build Features

Same feature set as the GNN: node features (GDP, exchange rate, inflation), edge features (log_imports, imports MoM%, lag exports 1–3), target (log_exports). Already filtered to the top 30 trading partners.

In [3]:
df = pd.read_csv(BASE / "data/final/master_trade_flow.csv", parse_dates=["date"])
df = df.sort_values(["country_imf", "date"]).reset_index(drop=True)

# Target
df["log_exports"] = np.log1p(df["exports_usd"])

# Edge features
df["log_imports"]     = np.log1p(df["imports_usd"])
df["imports_mom_pct"] = df.groupby("country_imf")["imports_usd"].pct_change() * 100

# Lag features
df["log_exports_lag1"] = df.groupby("country_imf")["log_exports"].shift(1)
df["log_exports_lag2"] = df.groupby("country_imf")["log_exports"].shift(2)
df["log_exports_lag3"] = df.groupby("country_imf")["log_exports"].shift(3)

df = df.replace([np.inf, -np.inf], np.nan)

NODE_FEATS = ["gdp_billions", "exchange_rate", "inflation"]
EDGE_FEATS = ["log_imports", "imports_mom_pct",
              "log_exports_lag1", "log_exports_lag2", "log_exports_lag3"]
TARGET     = "log_exports"

# Top 30 countries
top30 = df.groupby("country_imf")["exports_usd"].sum().nlargest(30).index.tolist()
df30  = df[df["country_imf"].isin(top30)].copy()
df30  = df30.sort_values(["country_imf", "date"]).reset_index(drop=True)

print(f"Shape     : {df30.shape}")
print(f"Countries : {df30['country_imf'].nunique()}")
print(f"Date range: {df30['date'].min().date()} → {df30['date'].max().date()}")
print(f"\nNaN counts (edge features):")
print(df30[EDGE_FEATS + [TARGET]].isna().sum())

Shape     : (2880, 23)
Countries : 30
Date range: 2017-01-01 → 2024-12-01

NaN counts (edge features):
log_imports          0
imports_mom_pct     30
log_exports_lag1    30
log_exports_lag2    60
log_exports_lag3    90
log_exports          0
dtype: int64


### Build Monthly Graphs

Same star topology as the GNN: one US hub node, 30 partner nodes, one edge per country per month. This step creates the base graphs before they are grouped into temporal sequences.

In [4]:
def build_monthly_graphs(frame, node_feats, edge_feats, target):
    graphs, dates = [], []
    for dt, sub in frame.groupby("date"):
        sub = sub.sort_values("country_imf").reset_index(drop=True)
        if len(sub) < 30:
            continue
        partner_nodes = sub[node_feats].fillna(0).values.astype(np.float32)
        us_node = partner_nodes.mean(axis=0, keepdims=True)
        x = np.vstack([us_node, partner_nodes])          # (31, 3)

        k = len(sub)
        edge_index = np.array([[0]*k, list(range(1, k+1))], dtype=np.int64)
        edge_attr  = sub[edge_feats].fillna(0).values.astype(np.float32)  # (30, 5)
        y          = sub[target].fillna(0).values.astype(np.float32)       # (30,)

        g = {
            "x"          : x,
            "edge_index" : edge_index,
            "edge_attr"  : edge_attr,
            "y"          : y,
            "countries"  : sub["country_imf"].tolist(),
        }
        graphs.append(g)
        dates.append(dt)
    return graphs, dates

all_graphs, all_dates = build_monthly_graphs(df30, NODE_FEATS, EDGE_FEATS, TARGET)

print(f"Total monthly graphs : {len(all_graphs)}")
print(f"Date range           : {min(all_dates).date()} → {max(all_dates).date()}")
g0 = all_graphs[3]
print(f"\nSample graph:")
print(f"  x shape          : {g0['x'].shape}")
print(f"  edge_index shape : {g0['edge_index'].shape}")
print(f"  edge_attr shape  : {g0['edge_attr'].shape}")
print(f"  y shape          : {g0['y'].shape}")

Total monthly graphs : 96
Date range           : 2017-01-01 → 2024-12-01

Sample graph:
  x shape          : (31, 3)
  edge_index shape : (2, 30)
  edge_attr shape  : (30, 5)
  y shape          : (30,)


### Build Temporal Sequences — 6-Month Sliding Windows

ST-GCN creates sequences of 6 consecutive monthly graphs; such that the model takes January through to June and predicts July, then February through to July and predict August — and so on.

This gives the LSTM enough history to detect trends and seasonal shifts.

In [5]:
sequences, targets, meta = [], [], []

for i in range(len(all_dates) - SEQ_LEN):
    window = all_graphs[i : i + SEQ_LEN]       # 6 months of graphs
    target_graph = all_graphs[i + SEQ_LEN]     # month 7 to predict

    sequences.append(window)
    targets.append(target_graph["y"])
    meta.append({
        "date"      : all_dates[i + SEQ_LEN],
        "countries" : target_graph["countries"],
    })

print(f"Total sequences : {len(sequences)}")
print(f"Each sequence   : {SEQ_LEN} monthly graphs → predict month {SEQ_LEN+1}")
print(f"First target    : {meta[0]['date'].date()}")
print(f"Last target     : {meta[-1]['date'].date()}")
print(f"Target shape    : {targets[0].shape}")

# Train/test split — consistent with all other models
train_seqs, train_tgts, train_meta = [], [], []
test_seqs,  test_tgts,  test_meta  = [], [], []

for s, t, m in zip(sequences, targets, meta):
    if pd.Timestamp(m["date"]).year <= 2022:
        train_seqs.append(s); train_tgts.append(t); train_meta.append(m)
    else:
        test_seqs.append(s);  test_tgts.append(t);  test_meta.append(m)

print(f"\nTrain sequences : {len(train_seqs)}")
print(f"Test sequences  : {len(test_seqs)}")

Total sequences : 90
Each sequence   : 6 monthly graphs → predict month 7
First target    : 2017-07-01
Last target     : 2024-12-01
Target shape    : (30,)

Train sequences : 66
Test sequences  : 24


### Scale Features



In [6]:
all_train_x = np.vstack([g["x"]         for s in train_seqs for g in s])
all_train_e = np.vstack([g["edge_attr"]  for s in train_seqs for g in s])
all_train_y = np.concatenate([t          for t in train_tgts])

scaler_x = StandardScaler().fit(all_train_x)
scaler_e = StandardScaler().fit(all_train_e)
y_mean   = float(all_train_y.mean())
y_std    = float(all_train_y.std())

def scale_seq(seq):
    out = []
    for g in seq:
        out.append({
            "x"         : scaler_x.transform(g["x"]).astype(np.float32),
            "edge_index": g["edge_index"],
            "edge_attr" : scaler_e.transform(g["edge_attr"]).astype(np.float32),
            "y"         : ((g["y"] - y_mean) / y_std).astype(np.float32),
            "countries" : g["countries"],
        })
    return out

train_scaled = [scale_seq(s) for s in train_seqs]
test_scaled  = [scale_seq(s) for s in test_seqs]

print(f"y_mean : {y_mean:.4f}")
print(f"y_std  : {y_std:.4f}")
print(f"Train scaled : {len(train_scaled)} sequences")
print(f"Test scaled  : {len(test_scaled)} sequences")
print(f"Scaler fit on train only — no leakage ✓")

y_mean : 21.6305
y_std  : 0.9145
Train scaled : 66 sequences
Test scaled  : 24 sequences
Scaler fit on train only — no leakage ✓


## ST-GCN Model — Spatio-Temporal Graph Network

### ST-GCN Model Definition

For each time step in the 6-month window, GCN layers produce country embeddings. The embeddings are then stacked into a sequence and fed into an LSTM. The LSTM's final hidden state is concatenated with the current edge features and passed to a small dense head for the final prediction. This architecture handles both the spatial (who trades with whom) and temporal (how that changes over time) dimensions.

In [7]:
class SpatioTemporalGCN(nn.Module):
    def __init__(self, node_in=3, edge_in=5, hid=64, lstm_hid=32):
        super().__init__()
        self.conv1 = GCNConv(node_in, hid)
        self.conv2 = GCNConv(hid, hid)
        self.lstm  = nn.LSTM(
            input_size  = 2 * hid + edge_in,
            hidden_size = lstm_hid,
            num_layers  = 1,
            batch_first = True,
        )
        self.head = nn.Sequential(
            nn.Linear(lstm_hid, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 1),
        )

    def spatial(self, g):
        x  = torch.tensor(g["x"],          dtype=torch.float32).to(device)
        ei = torch.tensor(g["edge_index"],  dtype=torch.long).to(device)
        ea = torch.tensor(g["edge_attr"],   dtype=torch.float32).to(device)
        ei_bi = torch.cat([ei, ei.flip(0)], dim=1)
        h = F.relu(self.conv1(x, ei_bi))
        h = F.relu(self.conv2(h, ei_bi))
        src, dst = ei
        return torch.cat([h[src], h[dst], ea], dim=-1)  # (30, 2*hid+edge_in)

    def forward(self, seq):
        # seq = list of 6 graph dicts
        steps = torch.stack([self.spatial(g) for g in seq], dim=1)  # (30, 6, feat)
        out, _ = self.lstm(steps)      # (30, 6, lstm_hid)
        return self.head(out[:, -1, :]).squeeze(-1)  # (30,)

model = SpatioTemporalGCN(
    node_in  = len(NODE_FEATS),   # 3
    edge_in  = len(EDGE_FEATS),   # 5
    hid      = 64,
    lstm_hid = 32,
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"Model     : SpatioTemporalGCN")
print(f"Node in   : {len(NODE_FEATS)}")
print(f"Edge in   : {len(EDGE_FEATS)}")
print(f"GCN hid   : 64  |  LSTM hid: 32")
print(f"Params    : {params:,}")

Model     : SpatioTemporalGCN
Node in   : 3
Edge in   : 5
GCN hid   : 64  |  LSTM hid: 32
Params    : 26,337


### Train the Model

150 epochs, Adam optimizer, gradient clipping. Since the LSTM processes 6-month windows, training takes longer per epoch than the regular GNN but learns richer patterns. The second training cell (epochs 101–150) continues from where the first left off.

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(seqs, tgts, train=True):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    for seq, tgt in zip(seqs, tgts):
        y = torch.tensor(tgt, dtype=torch.float32).to(device)
        if train:
            optimizer.zero_grad()
            pred = model(seq)
            loss = F.mse_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        else:
            with torch.no_grad():
                pred = model(seq)
                loss = F.mse_loss(pred, y)
        total_loss += loss.item() * len(tgt)
        n += len(tgt)
    return total_loss / n

for epoch in range(1, 101):
    tr = run_epoch(train_scaled, train_tgts, train=True)
    te = run_epoch(test_scaled,  test_tgts,  train=False)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train: {tr:.4f} | test: {te:.4f}")

Epoch  10 | train: 7.5249 | test: 0.4568
Epoch  20 | train: 7.4485 | test: 0.1434
Epoch  30 | train: 7.3034 | test: 0.1184
Epoch  40 | train: 6.5758 | test: 0.0677
Epoch  50 | train: 6.3322 | test: 0.0786
Epoch  60 | train: 6.3852 | test: 0.0462
Epoch  70 | train: 5.7857 | test: 0.0896
Epoch  80 | train: 5.4520 | test: 0.0451
Epoch  90 | train: 5.0167 | test: 0.0541
Epoch 100 | train: 5.0680 | test: 0.1300


In [9]:
# Scale targets (fit on train only — already have y_mean, y_std)
train_tgts_scaled = [(t - y_mean) / y_std for t in train_tgts]
test_tgts_scaled  = [(t - y_mean) / y_std for t in test_tgts]

# Reinitialize model fresh
torch.manual_seed(42)
model = SpatioTemporalGCN(
    node_in=len(NODE_FEATS), edge_in=len(EDGE_FEATS),
    hid=64, lstm_hid=32
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 151):
    tr = run_epoch(train_scaled, train_tgts_scaled, train=True)
    te = run_epoch(test_scaled,  test_tgts_scaled,  train=False)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train: {tr:.4f} | test: {te:.4f}")

Epoch  10 | train: 0.0635 | test: 0.0411
Epoch  20 | train: 0.0543 | test: 0.0366
Epoch  30 | train: 0.0515 | test: 0.0252
Epoch  40 | train: 0.0538 | test: 0.0514
Epoch  50 | train: 0.0477 | test: 0.0267
Epoch  60 | train: 0.0415 | test: 0.0288
Epoch  70 | train: 0.0421 | test: 0.0276
Epoch  80 | train: 0.0408 | test: 0.0271
Epoch  90 | train: 0.0441 | test: 0.0332
Epoch 100 | train: 0.0426 | test: 0.0274
Epoch 110 | train: 0.0420 | test: 0.0322
Epoch 120 | train: 0.0397 | test: 0.0407
Epoch 130 | train: 0.0405 | test: 0.0304
Epoch 140 | train: 0.0355 | test: 0.0292
Epoch 150 | train: 0.0368 | test: 0.0289


### Generate Predictions and Flag Anomalies

Running the trained model on the test set (2023–2024) and computing residuals (actual minus predicted), then apply per-country z-score thresholding: any month where |z| > 2.0 gets flagged as an anomaly.

Using per-country thresholds means a volatile country like Mexico and a stable country like Canada are judged against their own normal range, not a global one.

In [10]:
model.eval()
rows = []

with torch.no_grad():
    for seq, tgt, m in zip(test_scaled, test_tgts_scaled, test_meta):
        pred_z = model(seq).cpu().numpy()
        y_z    = tgt

        # Denormalize
        pred   = pred_z * y_std + y_mean
        y_true = y_z   * y_std + y_mean

        for country, yt, p in zip(m["countries"], y_true, pred):
            rows.append({
                "date"            : m["date"],
                "country_imf"     : country,
                "y_log_exports"   : float(yt),
                "pred_log_exports": float(p),
                "residual"        : float(yt - p),
            })

results = pd.DataFrame(rows)
results["residual_z"] = results.groupby("country_imf")["residual"].transform(
    lambda x: (x - x.mean()) / x.std()
)
results["stgcn_anomaly"] = results["residual_z"].abs() > 2.0

print(f"Test rows      : {len(results)}")
print(f"Countries      : {results['country_imf'].nunique()}")
print(f"Date range     : {results['date'].min().date()} → {results['date'].max().date()}")
print(f"ST-GCN anomalies: {results['stgcn_anomaly'].sum()}")
print(f"\nAnomalies per country:")
print(results[results['stgcn_anomaly']].groupby("country_imf").size().sort_values(ascending=False))

Test rows      : 720
Countries      : 30
Date range     : 2023-01-01 → 2024-12-01
ST-GCN anomalies: 31

Anomalies per country:
country_imf
Brazil                         3
United Arab Emirates           2
Switzerland                    2
Germany                        2
Belgium                        2
Australia                      1
Korea, Republic Of             1
United Kingdom                 1
Thailand                       1
Spain                          1
Singapore                      1
Saudi Arabia                   1
Netherlands                    1
Malaysia                       1
Italy                          1
Japan                          1
Israel                         1
Ireland                        1
France                         1
Dominican Republic             1
Colombia                       1
China, People'S Republic Of    1
Chile                          1
Canada                         1
Vietnam                        1
dtype: int64


### Results — Anomaly Counts and Performance Metrics

Summary of how many anomalies each country has in the test period, plus the overall R², MAE, and RMSE for the predictions.

In [11]:
# Fixing Turkey encoding
results["country_imf"] = results["country_imf"].str.replace(
    "Tã¼Rkiye, Republic Of", "Turkiye, Republic Of", regex=False
)

out_path = BASE / "data/final/stgcn_anomaly_results.csv"
results.to_csv(out_path, index=False)

print(f"Shape : {results.shape}")
print(f"\nFINAL SUMMARY")
print(f"Model       : SpatioTemporalGCN (no leakage)")
print(f"Node feats  : {NODE_FEATS}")
print(f"Edge feats  : {EDGE_FEATS}")
print(f"SEQ_LEN     : {SEQ_LEN} months")
print(f"Train       : 2017–2022 (66 sequences)")
print(f"Test        : 2023–2024 (24 sequences × 30 countries)")
print(f"Params      : 26,337")
print(f"Test MSE    : ~0.028 (z-scored)")
print(f"Anomalies   : {results['stgcn_anomaly'].sum()} across {results[results['stgcn_anomaly']]['country_imf'].nunique()} countries")


Shape : (720, 7)

FINAL SUMMARY
Model       : SpatioTemporalGCN (no leakage)
Node feats  : ['gdp_billions', 'exchange_rate', 'inflation']
Edge feats  : ['log_imports', 'imports_mom_pct', 'log_exports_lag1', 'log_exports_lag2', 'log_exports_lag3']
SEQ_LEN     : 6 months
Train       : 2017–2022 (66 sequences)
Test        : 2023–2024 (24 sequences × 30 countries)
Params      : 26,337
Test MSE    : ~0.028 (z-scored)
Anomalies   : 31 across 25 countries


### V2 — Reload Data with Distance Feature

Reloading the master file (which now includes distance_km and log_distance from the Week 7 distance fix), then rebuilding all monthly graphs and sequences with log_distance added as a 4th node feature.

In [12]:
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import StandardScaler


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "final").is_dir():
        return cwd
    if (cwd.parent / "data" / "final").is_dir():
        return cwd.parent
    return cwd


BASE = _repo_root()
torch.manual_seed(42); np.random.seed(42)
device = torch.device("cpu")
SEQ_LEN = 6

df = pd.read_csv(BASE / "data/final/master_trade_flow.csv", parse_dates=["date"])
df = df.sort_values(["country_imf", "date"]).reset_index(drop=True)
df["log_exports"]      = np.log1p(df["exports_usd"])
df["log_imports"]      = np.log1p(df["imports_usd"])
df["imports_mom_pct"]  = df.groupby("country_imf")["imports_usd"].pct_change() * 100
df["log_exports_lag1"] = df.groupby("country_imf")["log_exports"].shift(1)
df["log_exports_lag2"] = df.groupby("country_imf")["log_exports"].shift(2)
df["log_exports_lag3"] = df.groupby("country_imf")["log_exports"].shift(3)
df = df.replace([np.inf, -np.inf], np.nan)

if "log_distance" not in df.columns:
    if "distance_km" in df.columns:
        df["log_distance"] = np.log(df["distance_km"].replace(0, np.nan))
    else:
        dist_path = BASE / "data/distance_from_dc.csv"
        if not dist_path.exists():
            raise FileNotFoundError(
                f"Missing distance columns in master and file not found: {dist_path}\n"
                "Run notebooks/Week7_Distance_Fix.ipynb first, or keep data/distance_from_dc.csv."
            )
        dist = pd.read_csv(dist_path).rename(columns={"country": "country_exp"})
        df = df.merge(
            dist[["country_exp", "distance_km", "log_distance"]],
            on="country_exp",
            how="left",
            validate="many_to_one",
        )

NODE_FEATS = ["gdp_billions", "exchange_rate", "inflation", "log_distance"]
EDGE_FEATS = ["log_imports", "imports_mom_pct",
              "log_exports_lag1", "log_exports_lag2", "log_exports_lag3"]
TARGET = "log_exports"

top30 = df.groupby("country_imf")["exports_usd"].sum().nlargest(30).index.tolist()
df30  = df[df["country_imf"].isin(top30)].sort_values(["country_imf","date"]).reset_index(drop=True)

print(f"Rows: {len(df30)} | Countries: {df30['country_imf'].nunique()}")
print(f"NaN in log_distance: {df30['log_distance'].isna().sum()}")
print(f"NODE_FEATS: {NODE_FEATS}")

Rows: 2880 | Countries: 30
NaN in log_distance: 0
NODE_FEATS: ['gdp_billions', 'exchange_rate', 'inflation', 'log_distance']


## ST-GCN Update — Adding Distance as a Node Feature


### Build Sequences and Scale with 4 Node Features

Same sliding window logic as before, but now each graph has 4 node features instead of 3. Scalers are refit on the new training data to account for the added distance dimension.

In [13]:
def build_monthly_graphs(frame, node_feats, edge_feats, target):
    graphs, dates = [], []
    for dt, sub in frame.groupby("date"):
        sub = sub.sort_values("country_imf").reset_index(drop=True)
        if len(sub) < 30: continue
        partner_nodes = sub[node_feats].fillna(0).values.astype(np.float32)
        us_node = partner_nodes.mean(axis=0, keepdims=True)
        x = np.vstack([us_node, partner_nodes])
        k = len(sub)
        g = {"x": x,
             "edge_index": np.array([[0]*k, list(range(1,k+1))], dtype=np.int64),
             "edge_attr" : sub[edge_feats].fillna(0).values.astype(np.float32),
             "y"         : sub[target].fillna(0).values.astype(np.float32),
             "countries" : sub["country_imf"].tolist()}
        graphs.append(g); dates.append(dt)
    return graphs, dates

all_graphs, all_dates = build_monthly_graphs(df30, NODE_FEATS, EDGE_FEATS, TARGET)

sequences, targets, meta = [], [], []
for i in range(len(all_dates) - SEQ_LEN):
    sequences.append(all_graphs[i:i+SEQ_LEN])
    targets.append(all_graphs[i+SEQ_LEN]["y"])
    meta.append({"date": all_dates[i+SEQ_LEN],
                 "countries": all_graphs[i+SEQ_LEN]["countries"]})

train_seqs = [s for s,m in zip(sequences,meta) if pd.Timestamp(m["date"]).year<=2022]
train_tgts = [t for t,m in zip(targets,meta)   if pd.Timestamp(m["date"]).year<=2022]
train_meta = [m for m in meta                   if pd.Timestamp(m["date"]).year<=2022]
test_seqs  = [s for s,m in zip(sequences,meta) if pd.Timestamp(m["date"]).year>=2023]
test_tgts  = [t for t,m in zip(targets,meta)   if pd.Timestamp(m["date"]).year>=2023]
test_meta  = [m for m in meta                   if pd.Timestamp(m["date"]).year>=2023]

all_x = np.vstack([g["x"]        for s in train_seqs for g in s])
all_e = np.vstack([g["edge_attr"] for s in train_seqs for g in s])
all_y = np.concatenate([t for t in train_tgts])
scaler_x = StandardScaler().fit(all_x)
scaler_e = StandardScaler().fit(all_e)
y_mean, y_std = float(all_y.mean()), float(all_y.std())

def scale_seq(seq):
    return [{"x": scaler_x.transform(g["x"]).astype(np.float32),
             "edge_index": g["edge_index"],
             "edge_attr": scaler_e.transform(g["edge_attr"]).astype(np.float32),
             "y": ((g["y"]-y_mean)/y_std).astype(np.float32),
             "countries": g["countries"]} for g in seq]

train_scaled = [scale_seq(s) for s in train_seqs]
test_scaled  = [scale_seq(s) for s in test_seqs]
train_tgts_s = [(t-y_mean)/y_std for t in train_tgts]
test_tgts_s  = [(t-y_mean)/y_std for t in test_tgts]

print(f"Train: {len(train_scaled)} | Test: {len(test_scaled)}")
print(f"Node features: {len(NODE_FEATS)} (including log_distance)")

Train: 66 | Test: 24
Node features: 4 (including log_distance)


### ST-GCN V2 — node_in=4, Train with Distance

Same architecture — just the input dimension updated to 4. Training uses 150 epochs to give the model more time to pick up the distance signal alongside the time-varying features.

In [14]:
class SpatioTemporalGCN(nn.Module):
    def __init__(self, node_in=4, edge_in=5, hid=64, lstm_hid=32):
        super().__init__()
        self.conv1 = GCNConv(node_in, hid)
        self.conv2 = GCNConv(hid, hid)
        self.lstm  = nn.LSTM(input_size=2*hid+edge_in,
                             hidden_size=lstm_hid, batch_first=True)
        self.head  = nn.Sequential(nn.Linear(lstm_hid,16), nn.ReLU(),
                                   nn.Dropout(0.1), nn.Linear(16,1))

    def spatial(self, g):
        x  = torch.tensor(g["x"],         dtype=torch.float32).to(device)
        ei = torch.tensor(g["edge_index"], dtype=torch.long).to(device)
        ea = torch.tensor(g["edge_attr"],  dtype=torch.float32).to(device)
        ei_bi = torch.cat([ei, ei.flip(0)], dim=1)
        h = F.relu(self.conv1(x, ei_bi))
        h = F.relu(self.conv2(h, ei_bi))
        src, dst = ei
        return torch.cat([h[src], h[dst], ea], dim=-1)

    def forward(self, seq):
        steps = torch.stack([self.spatial(g) for g in seq], dim=1)
        out, _ = self.lstm(steps)
        return self.head(out[:,-1,:]).squeeze(-1)

model_st = SpatioTemporalGCN(node_in=4, edge_in=5, hid=64, lstm_hid=32).to(device)
optimizer_st = torch.optim.Adam(model_st.parameters(), lr=1e-3)

def run_epoch_st(seqs, tgts, train=True):
    model_st.train() if train else model_st.eval()
    total, n = 0.0, 0
    for seq, tgt in zip(seqs, tgts):
        y = torch.tensor(tgt, dtype=torch.float32).to(device)
        if train:
            optimizer_st.zero_grad()
            pred = model_st(seq)
            loss = F.mse_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_st.parameters(), 1.0)
            optimizer_st.step()
        else:
            with torch.no_grad():
                pred = model_st(seq); loss = F.mse_loss(pred, y)
        total += loss.item()*len(tgt); n += len(tgt)
    return total/n

for epoch in range(1, 151):
    tr = run_epoch_st(train_scaled, train_tgts_s, train=True)
    te = run_epoch_st(test_scaled,  test_tgts_s,  train=False)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train: {tr:.4f} | test: {te:.4f}")

Epoch  10 | train: 0.0651 | test: 0.0613
Epoch  20 | train: 0.0520 | test: 0.0350
Epoch  30 | train: 0.0530 | test: 0.0470
Epoch  40 | train: 0.0469 | test: 0.0270
Epoch  50 | train: 0.0462 | test: 0.0263
Epoch  60 | train: 0.0430 | test: 0.0302
Epoch  70 | train: 0.0446 | test: 0.0279
Epoch  80 | train: 0.0436 | test: 0.0399
Epoch  90 | train: 0.0415 | test: 0.0286
Epoch 100 | train: 0.0403 | test: 0.0301
Epoch 110 | train: 0.0407 | test: 0.0327
Epoch 120 | train: 0.0384 | test: 0.0339
Epoch 130 | train: 0.0438 | test: 0.0403
Epoch 140 | train: 0.0391 | test: 0.0324
Epoch 150 | train: 0.0399 | test: 0.0288


### V2 Predictions and Final Save

In [15]:
model_st.eval()
rows = []
with torch.no_grad():
    for seq, tgt, m in zip(test_scaled, test_tgts_s, test_meta):
        pred = model_st(seq).cpu().numpy() * y_std + y_mean
        ytrue = np.array(tgt)             * y_std + y_mean
        for country, yt, p in zip(m["countries"], ytrue, pred):
            rows.append({"date": m["date"], "country_imf": country,
                         "y_log_exports": float(yt),
                         "pred_log_exports": float(p),
                         "residual": float(yt - p)})

res = pd.DataFrame(rows)
res["residual_z"]    = res.groupby("country_imf")["residual"].transform(
    lambda x: (x - x.mean()) / x.std())
res["stgcn_anomaly"] = res["residual_z"].abs() > 2.0
res["country_imf"]   = res["country_imf"].str.replace(
    "Tã¼Rkiye, Republic Of", "Turkiye, Republic Of", regex=False)

res.to_csv(BASE / "data/final/stgcn_anomaly_results.csv", index=False)

print(f"Anomalies: {res['stgcn_anomaly'].sum()} across {res[res['stgcn_anomaly']]['country_imf'].nunique()} countries")
print(f"Node features: {NODE_FEATS}")

Anomalies: 30 across 24 countries
Node features: ['gdp_billions', 'exchange_rate', 'inflation', 'log_distance']
